# MobApp Studio, workshop 1 — Cleaning is a chain of decisions

**The situation.** John, your boss, wants a market report on the Google Play Store before the
company commits a team to building an app. He asks for the size of the market in downloads and
dollars, the split per category, and the downloads-per-app ratio.

You have one file of scraped apps. Before any of his questions can be answered, the file has to
become trustworthy — and **every cleaning step you take is a decision you will have to defend**,
first to John's VPs, then in the blog post the project asks you to publish.

Today we find out what is wrong with the file, and we measure what each fix is worth.

The headline, which you will reproduce yourself in exercise 2: skipping one of these steps
overstates the market by **more than double**.

In [ ]:
# ABOUTME: MobApp Studio workshop 1 - auditing and cleaning the Google Play Store export.
# ABOUTME: Finds the corrupt row, measures the duplicate problem, and parses text columns to numbers.

from pathlib import Path
import numpy as np
import pandas as pd

def qwasar_file(name):
    """Locate a dataset fetched by workshops/get_data.py, from wherever this notebook was opened."""
    for up in range(4):
        p = Path(*([".."] * up), "data", name)
        if p.exists():
            return p
    raise FileNotFoundError(
        f"{name} is missing. Run `uv run get_data.py` in the workshops/ folder first - "
        "these notebooks only ever use the real dataset the project links to.")

# The Google Play export the project points at. 10,841 rows, exactly as published.
raw = pd.read_csv(qwasar_file("googleplaystore.csv"))

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 20)
print(f"loaded {len(raw):,} rows x {raw.shape[1]} columns from the real Google Play export")


## 1. Read the types before you read the data

`print_summarize_dataset` is the first function the project asks for, and the most useful line it
prints is the list of dtypes. Pandas reports what it *managed* to parse — so a column of numbers
that arrives as `object` is telling you something is wrong with it.

In [ ]:
print("shape:", raw.shape)
print("\ntypes:\n", raw.dtypes.to_string())
print("\nmissing values:\n", raw.isna().sum()[lambda s: s > 0].to_string())
raw.head(3)

Look at what is `object` (that is pandas for "text"): **Reviews, Size, Installs, Price**. Four
columns you will need to add up, and not one of them is a number yet.

And look at the sample values: `"10,000+"`, `"19M"`, `"$4.99"`, `"Varies with device"`. Each one is
a formatting choice made by whoever built the scraper, and each needs a decision from you.

**Rule of thumb:** `df.dtypes` is a to-do list. Every `object` column you plan to compute with is
one item on it.

## 2. One corrupt row, five symptoms

Before parsing anything, look for values that cannot exist. A rating is out of 5. An install count
is a number. A price starts with a dollar sign or is zero.

In [ ]:
print("Rating      : min", raw.Rating.min(), " max", raw.Rating.max(), " <- out of 5?")
print("Installs    : values that are not digits/commas/plus ->",
      [v for v in raw.Installs.unique() if not str(v).replace(",", "").replace("+", "").isdigit()])
print("Price       : values that are neither $x nor 0 ->",
      sorted({v for v in raw.Price.unique() if not str(v).startswith("$") and str(v) != "0"}))
print("Reviews     : values that are not plain digits ->",
      [v for v in raw.Reviews.unique() if not str(v).isdigit()][:5])
print("Type        :", raw.Type.value_counts(dropna=False).to_dict())

Five separate anomalies. A beginner writes five `try/except` blocks. A data scientist asks whether
they have **one** cause.

### Exercise 1 — find the single row behind all of it

Category is a controlled vocabulary: `GAME`, `FAMILY`, `TOOLS`... always capitals and underscores.
Find the rows whose Category does not look like that, and print them in full.

In [ ]:
def find_corrupt_rows(df):
    """Return the rows whose Category is not an ALL_CAPS label."""
    # TODO: use df.Category.astype(str).str.match(...) with a regex for ^[A-Z_]+$
    #       and return the rows that do NOT match
    return df.iloc[0:0]

corrupt = find_corrupt_rows(raw)
print(f"{len(corrupt)} corrupt row(s)")
corrupt

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("finds exactly one row", lambda: len(corrupt) == 1)
check("it is the one with the impossible rating", lambda: float(corrupt.Rating.iloc[0]) > 5)
check("every other row has a clean category",
      lambda: raw.drop(corrupt.index).Category.astype(str).str.match(r"^[A-Z_]+$").all())

There it is. Read the row left to right and you can see what happened: the **Category is missing**,
so every value after it slid one column to the left. The rating is really a category, the "reviews"
value is really a size, `Installs` holds `Free`, `Price` holds `Everyone`.

One broken row, five symptoms, and it accounts for **all** of them.

**Rule of thumb:** before writing five fixes, check whether you have one problem. Anomalies that
appear together usually share a cause.

What to do with it? You have three options, and you must pick one and say why:

1. **Repair it** — shift the values back one column. Tempting, but the Category it lost is not
   recoverable; you would be inventing the one field you cannot infer.
2. **Drop it** — one row in ten thousand, with no way to know what it should have said.
3. **Keep it** — and let a rating of 19 into every statistic you publish.

Two is the defensible choice here, and the blog post says so in one line: *"one row (10472) was
malformed — its category was missing and the remaining fields were shifted; it was removed."*

## 3. The duplicate problem, and what it costs

The scraper visited some apps more than once. The rows are not errors — each was true on the day
it was collected — but adding them all up counts the same installs repeatedly.

In [ ]:
clean_cat = raw[raw.Category.astype(str).str.match(r"^[A-Z_]+$")].copy()
print("rows after dropping the corrupt one :", len(clean_cat))
print("exact duplicate rows                :", clean_cat.duplicated().sum())
print("rows sharing an App name            :", clean_cat.App.duplicated().sum())
print("\nan app that appears several times:")
repeated = clean_cat[clean_cat.App.duplicated(keep=False)].sort_values("App")
repeated.head(4)[["App", "Category", "Rating", "Reviews", "Installs"]]

Notice the two numbers differ. **Exact duplicates** are byte-identical rows. **Repeated app names**
include scrapes taken days apart, where the review count moved. Which definition you use changes
how many rows you keep, so it is another decision to declare.

A reasonable policy: one row per app, keeping the one with the **most reviews** — the most recent
observation.

### Exercise 2 — measure what the decision is worth

Parse `Installs` into numbers, then compare the total before and after de-duplication. This single
number is the most important sentence in your report.

In [ ]:
def parse_installs(series):
    """'10,000+' -> 10000, as an integer. Non-numeric values become NaN."""
    # TODO: strip the commas and the plus sign, then pd.to_numeric(..., errors="coerce")
    return pd.Series(dtype="float64")

def deduplicate(df):
    """One row per app: keep the one with the most Reviews."""
    # TODO 1: make Reviews numeric (errors="coerce")
    # TODO 2: sort by Reviews descending, then drop_duplicates on "App" keeping the first
    return df

with_dupes = clean_cat.assign(n=parse_installs(clean_cat.Installs))
without = deduplicate(clean_cat).assign(n=lambda d: parse_installs(d.Installs))
total_dupes, total_clean = with_dupes.n.sum(), without.n.sum()
print(f"with duplicates : {total_dupes:,.0f} installs over {len(with_dupes):,} rows")
print(f"de-duplicated   : {total_clean:,.0f} installs over {len(without):,} rows")

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("installs parsed to numbers", lambda: float(total_clean) > 0)
check("de-duplication removed rows", lambda: len(without) < len(with_dupes))
check("one row per app now", lambda: not without.App.duplicated().any())
check("and the total dropped", lambda: total_clean < total_dupes)

On the real export that is **+123 %**: 167.6 billion installs claimed against 75.3 billion real
ones. You would walk into John's meeting and describe a market more than twice its actual size,
with a chart to prove it, and nothing in the data would have warned you.

**The punchline:** the most expensive mistakes in data work are not crashes. They are numbers that
look plausible.

## 4. Prices, and the missing ratings

Two more columns to deal with, and the second is subtler than it looks.

In [ ]:
def parse_price(series):
    """'$4.99' -> 4.99, '0' -> 0.0."""
    # TODO: remove the dollar sign, then to_numeric with errors="coerce"
    return pd.Series(dtype="float64")

df = without.copy()
df["Price"] = parse_price(df.Price)
df["Installs"] = parse_installs(df.Installs)
print("paid apps:", int((df.Price > 0).sum()), "| free apps:", int((df.Price == 0).sum()))
print("most expensive:", df.Price.max())

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("prices are numeric", lambda: df.Price.dtype.kind == "f")
check("free apps are zero, not NaN", lambda: (df.Price == 0).sum() > 0)
check("someone is selling an app for a lot of money", lambda: df.Price.max() > 50)

That last check should surprise you — we come back to it next workshop, because it quietly ruins
one of the charts John asked for.

### Exercise 3 — are the missing ratings missing *at random*?

Plenty of apps have no rating. The reflex is `dropna()`. Before you do, ask **why** they are
missing, because that determines whether dropping them is harmless or biasing.

Compare the apps that have a rating with the ones that do not, on a column you still trust.

In [ ]:
rated = df[df.Rating.notna()]
unrated = df[df.Rating.isna()]
reviews = pd.to_numeric(df.Reviews, errors="coerce")

# TODO: compare the two groups on their number of reviews (median is the robust choice)
median_reviews_rated = None
median_reviews_unrated = None
print("median reviews, rating present:", median_reviews_rated)
print("median reviews, rating missing:", median_reviews_unrated)

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("both medians computed", lambda: median_reviews_rated is not None and median_reviews_unrated is not None)
check("unrated apps have far fewer reviews", lambda: median_reviews_unrated < median_reviews_rated)

On the real file the gap is brutal: a median of **3 017 reviews** for apps that have a rating,
against **1** for apps that do not. The rating is not missing by accident — it is missing *because*
nobody used the app.

That is called **missing not at random**, and it has a direct consequence: `dropna()` does not
remove noise, it removes the unpopular half of the store. Any sentence you then write about "the
average app" is really about the average *successful* app.

What to do:
- For install and price totals: **keep** the rows. A missing rating does not make the installs
  unknown.
- For anything averaging ratings: drop them, and **say so**, with the number of apps excluded.

**Rule of thumb:** `dropna()` is a decision about who is in your study. Never make it silently.

## What to take away

1. **`dtypes` is your to-do list.** Numbers stored as text are the project's main obstacle.
2. **Look for one cause before writing five fixes.** One shifted row produced every impossible
   value in this file.
3. **Say what you did with corrupt rows** — repair, drop or keep — and why. There is no neutral
   option, only documented and undocumented ones.
4. **De-duplicate before you sum anything.** Here it is the difference between 75 billion installs
   and 168 billion.
5. **Missing is information.** Check whether missingness correlates with something before dropping.
6. Every one of these is a line in your blog post. Write them down as you go; reconstructing them
   the night before the review is how details get lost.

### Take it home
Assemble these steps into the `clean_dataset(dataset)` the project asks for, and have it print a
short before/after summary: rows in, rows out, and why. That printout is the backbone of your
report, and the function is one of the six you must submit.